# Experiment 5 & 6 Rerun — Final HMRNet Stage 8 v2

Notebook ini menjalankan ulang:

- **Experiment 5 — Downstream Utility**
- **Experiment 6 — Automatic Semantic Interpretability**

menggunakan checkpoint final:

`results/checkpoints/HMRNet-Stage8-v2-CV-GT070.pt`

## Tujuan

Semua hasil HMRNet pada manuscript final harus merujuk pada konfigurasi yang sama. Experiment 5 menggunakan topic representation final untuk sentiment classification, sedangkan Experiment 6 menghitung WIS, TIS, Semantic Compactness, dan Inter-topic Separation dari topic structure final.

### Experiment 5
**Label policy:** sentiment labels are automatically recovered from the same source used by the previous Experiment 5 whenever that source/artifact is still present in the project. The notebook does **not** ask for, generate, or infer new sentiment labels.

Primary protocol:
- fixed stratified 80:20 split;
- Transformer-only;
- Transformer + HMRNet-Final;
- LDA / BERTopic / Top2Vec dipakai kembali bila document-topic representation lama ditemukan;
- jika tabel Experiment 05 lama ditemukan tetapi vector baseline tidak ditemukan, tabel lama digunakan hanya untuk **unchanged baseline reference**, dengan source ditandai eksplisit.

Tambahan Q1:
- 10 repeated stratified splits untuk Transformer-only vs Transformer + HMRNet;
- paired Wilcoxon dan paired t-test pada Macro-F1.

### Experiment 6
- Word Intrusion Score (WIS)
- Topic Intrusion Score (TIS)
- Semantic Compactness (SC)
- Inter-topic Separation (IS)

WIS/TIS/SC/IS adalah **automatic semantic proxies**, bukan pengganti human evaluation.

## 1. Install libraries

In [ ]:
%pip install -q "gensim==4.4.0" openpyxl sentence-transformers scipy scikit-learn joblib

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Imports and project discovery

In [ ]:
from pathlib import Path
from collections import defaultdict, deque
import sys
import gc
import json
import random
import re
import inspect
import warnings

import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from scipy import sparse
from scipy.stats import wilcoxon, ttest_rel

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer
from IPython.display import display

PROJECT_CANDIDATES = [
    Path("/content/drive/MyDrive/UDB/Riset/2026"),
]

PROJECT_ROOT = None
for candidate in PROJECT_CANDIDATES:
    if candidate.exists() and (candidate / "src" / "hmrnet.py").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project HMRNet tidak ditemukan. "
        "Perbaiki PROJECT_CANDIDATES sesuai lokasi Google Drive."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import load_feature_bundle
from src.hmrnet import HMRNet

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"
FEATURE_DIR = CHECKPOINTS_DIR / "features"

EXP5_DIR = RESULTS_DIR / "experiment_05_stage8v2"
EXP6_DIR = RESULTS_DIR / "experiment_06_stage8v2"

for d in [TABLES_DIR, FIGURES_DIR, EXP5_DIR, EXP6_DIR]:
    d.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Configuration

In [ ]:
FINAL_HMRNET_CHECKPOINT = (
    CHECKPOINTS_DIR / "HMRNet-Stage8-v2-CV-GT070.pt"
)

if not FINAL_HMRNET_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Checkpoint final tidak ditemukan: {FINAL_HMRNET_CHECKPOINT}\n"
        "Pastikan Stage 8 v2 selesai dan checkpoint sudah disimpan."
    )

# Fixed-split primary Experiment 5
PRIMARY_SPLIT_SEED = 42
TEST_SIZE = 0.20

# Additional Q1 repeated-split validation
RUN_REPEATED_SPLITS = True
REPEATED_SPLIT_SEEDS = list(range(1, 11))

LOGREG_MAX_ITER = 5000
LOGREG_CLASS_WEIGHT = "balanced"

# Baseline topic representation handling
AUTO_DISCOVER_BASELINE_VECTORS = True
FIT_LDA_IF_MISSING = True

# BERTopic/Top2Vec are NOT retrained automatically because they are expensive
# and may not reproduce the exact original Experiment-01 models.
RETRAIN_BERTOPIC_IF_MISSING = False
RETRAIN_TOP2VEC_IF_MISSING = False

# Interpretability
TOP_N_WORDS = 10
SEMANTIC_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
WIS_REPEATS = 20
WIS_RANDOM_SEED = 42
TIS_RANDOM_SEED = 42
TIS_N_DISTRACTORS = 4
INTERPRETABILITY_DOC_LIMIT = None  # None = all documents

print("Final checkpoint:", FINAL_HMRNET_CHECKPOINT)

## 5. Load the fixed feature bundle

In [ ]:
bundle = load_feature_bundle(FEATURE_DIR)

print("Feature bundle keys:")
print(sorted(bundle.keys()))

raw_texts = list(bundle["raw_texts"])
processed_texts = list(bundle["processed_texts"])
bow = bundle["bow"]
semantic_embeddings = np.asarray(bundle["semantic_embeddings"])
document_embeddings = np.asarray(bundle["document_embeddings"])
vocabulary = list(bundle["vocabulary"])

n_documents = len(processed_texts)

assert len(raw_texts) == n_documents
assert bow.shape[0] == n_documents
assert semantic_embeddings.shape[0] == n_documents
assert document_embeddings.shape[0] == n_documents

print("\nDocuments:", n_documents)
print("Vocabulary:", len(vocabulary))
print("BoW:", bow.shape)
print("Semantic embeddings:", semantic_embeddings.shape)
print("Document embeddings:", document_embeddings.shape)

## 6. Load final Stage 8 v2 HMRNet

In [ ]:
hmrnet = HMRNet.load(
    FINAL_HMRNET_CHECKPOINT,
    device=DEVICE,
)

print("Loaded:", FINAL_HMRNET_CHECKPOINT)

if hasattr(hmrnet, "config"):
    print("\nFinal HMRNet configuration:")
    for key, value in sorted(vars(hmrnet.config).items()):
        if not key.startswith("_"):
            print(f"  {key}: {value}")

## 7. Extract final HMRNet document-topic representation

Cell ini dibuat adaptif terhadap variasi signature `HMRNet.transform()`. Output yang dipilih harus memiliki bentuk `(N, K)`. Jika output non-negative tetapi belum ternormalisasi, baris dinormalisasi menjadi distribusi topik.

In [ ]:
def _extract_matrix_from_transform_output(output, n_docs, expected_k=None):
    candidates = []

    if isinstance(output, dict):
        preferred_keys = [
            "theta",
            "document_topic_distribution",
            "topic_distribution",
            "topic_proportions",
            "topics",
            "z_topic",
            "latent",
            "embedding",
        ]
        for key in preferred_keys:
            if key in output:
                candidates.append((key, output[key]))
        for key, value in output.items():
            if key not in preferred_keys:
                candidates.append((key, value))

    elif isinstance(output, (tuple, list)):
        for idx, value in enumerate(output):
            candidates.append((f"output_{idx}", value))
    else:
        candidates.append(("output", output))

    valid = []
    for name, value in candidates:
        try:
            arr = value.detach().cpu().numpy() if torch.is_tensor(value) else np.asarray(value)
        except Exception:
            continue

        if arr.ndim == 2 and arr.shape[0] == n_docs:
            score = 0
            if expected_k is not None and arr.shape[1] == expected_k:
                score += 10
            if np.all(np.isfinite(arr)):
                score += 1
            valid.append((score, name, arr))

    if not valid:
        raise RuntimeError(
            "Tidak ditemukan matrix document-topic berukuran (N, K) "
            "pada output HMRNet.transform()."
        )

    valid.sort(key=lambda item: item[0], reverse=True)
    _, name, arr = valid[0]
    print("Selected transform output:", name, arr.shape)
    return arr.astype(np.float32)


def get_hmrnet_topic_matrix(model, bow, semantic_embeddings, document_embeddings):
    if not hasattr(model, "transform"):
        raise AttributeError("HMRNet model tidak memiliki method transform().")

    expected_k = None
    if hasattr(model, "config") and hasattr(model.config, "n_topics"):
        expected_k = int(model.config.n_topics)

    attempts = [
        lambda: model.transform(
            bow=bow,
            semantic_embeddings=semantic_embeddings,
            document_embeddings=document_embeddings,
        ),
        lambda: model.transform(
            bow,
            semantic_embeddings,
            document_embeddings,
        ),
    ]

    last_error = None
    for i, attempt in enumerate(attempts, start=1):
        try:
            output = attempt()
            matrix = _extract_matrix_from_transform_output(
                output,
                n_docs=bow.shape[0],
                expected_k=expected_k,
            )
            print(f"transform attempt {i}: SUCCESS")
            break
        except Exception as exc:
            last_error = exc
            print(f"transform attempt {i}: {type(exc).__name__}: {exc}")
    else:
        raise RuntimeError(
            "Semua cara memanggil HMRNet.transform() gagal."
        ) from last_error

    # Convert to valid topic proportions when appropriate.
    row_sums = matrix.sum(axis=1, keepdims=True)

    if np.all(matrix >= -1e-8) and np.all(row_sums > 0):
        matrix = np.clip(matrix, 0, None)
        matrix = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1e-12)
    else:
        # Treat as logits/latent scores and apply row-wise softmax.
        shifted = matrix - matrix.max(axis=1, keepdims=True)
        exp_x = np.exp(shifted)
        matrix = exp_x / np.maximum(exp_x.sum(axis=1, keepdims=True), 1e-12)

    return matrix.astype(np.float32)


hmrnet_topic_matrix = get_hmrnet_topic_matrix(
    hmrnet,
    bow,
    semantic_embeddings,
    document_embeddings,
)

print("\nHMRNet topic matrix:", hmrnet_topic_matrix.shape)
print("Row-sum mean:", hmrnet_topic_matrix.sum(axis=1).mean())
print("Min/Max:", hmrnet_topic_matrix.min(), hmrnet_topic_matrix.max())

np.save(
    EXP5_DIR / "hmrnet_stage8v2_document_topic.npy",
    hmrnet_topic_matrix,
)


## 8. Automatically recover the previous Experiment-5 sentiment labels

**Important correction:** the previous Experiment 5 used a **separate annotated dataset (`ANNOTATED_PATH`)**, so the sentiment labels do **not** need to exist for all 23,370 topic-modeling documents. The previous auto-label version was too strict because it required a full-length label vector.

This version:

1. checks labels already stored in the HMRNet feature bundle;
2. searches the HMRNet project for previous annotation files;
3. searches the wider Google Drive for likely MBG / sentiment / annotation datasets;
4. accepts a **labeled subset**;
5. aligns annotated texts to the current HMRNet feature bundle;
6. creates a full-length label mask internally, but Experiment 5 uses **only the matched labeled documents**;
7. never fabricates sentiment labels.

This matches the earlier workflow in which `ANNOTATED_PATH` was loaded separately from the main corpus and then used for supervised sentiment evaluation.


In [ ]:
LABEL_KEYS = [
    "labels",
    "label",
    "sentiment",
    "sentiments",
    "sentiment_labels",
    "target",
    "targets",
    "y",
    "category",
]

TEXT_COLUMNS = [
    "text",
    "raw_text",
    "clean_text",
    "processed_text",
    "tweet",
    "tweet_text",
    "content",
    "document",
    "sentence",
    "teks",
    "kalimat",
]

LABEL_COLUMNS = [
    "sentimen",
    "sentiment",
    "sentiment_label",
    "sentiment_class",
    "label",
    "labels",
    "category",
    "target",
    "class",
    "kelas",
    "polaritas",
    "y",
]

SENTIMENT_ALIASES = {
    "negatif": "negative",
    "negative": "negative",
    "neg": "negative",
    "netral": "neutral",
    "neutral": "neutral",
    "neu": "neutral",
    "positif": "positive",
    "positive": "positive",
    "pos": "positive",
}

DRIVE_ROOT = Path("/content/drive/MyDrive")

EXCLUDED_LABEL_PATH_TOKENS = [
    "experiment_05_stage8v2",
    "05_06_hmrnet_stage8v2",
    "stage8v2",
]


def normalize_text_for_match(x):
    text = str(x)
    text = text.replace("\u00a0", " ")
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_text_basic_for_match(x):
    text = str(x).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", " ", text)
    text = re.sub(r"[^a-zA-ZÀ-ÿ\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_label_value(x):
    value = str(x).strip().lower()
    return SENTIMENT_ALIASES.get(value, value)


def is_excluded_label_path(path):
    low = str(path).lower()
    return any(token in low for token in EXCLUDED_LABEL_PATH_TOKENS)


def read_tabular(path):
    suffix = path.suffix.lower()

    if suffix == ".csv":
        last_error = None
        for encoding in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
            try:
                return pd.read_csv(path, encoding=encoding, low_memory=False)
            except Exception as exc:
                last_error = exc
        raise RuntimeError(f"Cannot read {path}: {last_error}")

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    if suffix == ".json":
        try:
            return pd.read_json(path)
        except Exception:
            with open(path, "r", encoding="utf-8") as f:
                obj = json.load(f)
            if isinstance(obj, list):
                return pd.DataFrame(obj)
            if isinstance(obj, dict):
                for key in ["data", "records", "items", "documents"]:
                    if key in obj and isinstance(obj[key], list):
                        return pd.DataFrame(obj[key])

    return None


def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )
    return df


def find_column(df, candidates):
    lower_map = {
        str(col).strip().lower(): col
        for col in df.columns
    }
    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]
    return None


# ------------------------------------------------------------------
# Build occurrence-aware indices for BOTH raw and processed HMRNet texts.
# This supports duplicate documents without silently changing order.
# ------------------------------------------------------------------

def build_feature_occurrence_index(texts, normalizer):
    index = defaultdict(deque)
    for i, text in enumerate(texts):
        key = normalizer(text)
        if key:
            index[key].append(i)
    return index


def align_annotated_subset_to_hmrnet(df, text_col, label_col):
    """
    Align a labeled subset to the HMRNet corpus.

    Returns:
        full_labels: object array length n_documents, None for unlabeled docs
        match_info: metadata dictionary
    """
    tmp = df[[text_col, label_col]].dropna().copy()

    if tmp.empty:
        return None

    tmp[text_col] = tmp[text_col].astype(str)
    tmp[label_col] = tmp[label_col].map(normalize_label_value)

    # Require at least two classes in source.
    tmp = tmp[
        ~tmp[label_col].isin(["", "nan", "none", "null"])
    ].copy()

    if tmp[label_col].nunique() < 2:
        return None

    strategies = [
        (
            "raw_exact_normalized",
            raw_texts,
            normalize_text_for_match,
        ),
        (
            "processed_exact_normalized",
            processed_texts,
            normalize_text_for_match,
        ),
        (
            "raw_basic_cleaning",
            raw_texts,
            normalize_text_basic_for_match,
        ),
        (
            "processed_basic_cleaning",
            processed_texts,
            normalize_text_basic_for_match,
        ),
    ]

    best = None

    for strategy_name, target_texts, normalizer in strategies:
        pools = build_feature_occurrence_index(
            target_texts,
            normalizer,
        )

        full_labels = np.empty(
            n_documents,
            dtype=object,
        )
        full_labels[:] = None

        matched = 0
        unmatched_source = 0
        conflicts = 0

        # Track labels already assigned to a document to avoid silent conflicts.
        for text, label in zip(
            tmp[text_col],
            tmp[label_col],
        ):
            key = normalizer(text)

            if not key or not pools[key]:
                unmatched_source += 1
                continue

            feature_idx = pools[key].popleft()

            if full_labels[feature_idx] is None:
                full_labels[feature_idx] = str(label)
                matched += 1
            elif full_labels[feature_idx] != str(label):
                conflicts += 1

        valid_values = [
            x for x in full_labels
            if x is not None
        ]

        if len(valid_values) == 0:
            continue

        n_classes = pd.Series(valid_values).nunique()

        candidate = {
            "strategy": strategy_name,
            "full_labels": full_labels,
            "matched": matched,
            "unmatched_source": unmatched_source,
            "conflicts": conflicts,
            "n_classes": int(n_classes),
            "source_rows": int(len(tmp)),
            "match_rate_source": (
                float(matched / len(tmp))
                if len(tmp)
                else 0.0
            ),
        }

        if (
            best is None
            or candidate["matched"] > best["matched"]
        ):
            best = candidate

    if best is None:
        return None

    # Minimum safeguards:
    # - at least 2 sentiment classes
    # - enough matched documents for a meaningful 80:20 stratified split
    if best["n_classes"] < 2:
        return None

    if best["matched"] < 30:
        return None

    return best


def labels_from_bundle():
    """
    Full labels already stored in feature bundle.
    """
    for key in LABEL_KEYS:
        if key not in bundle:
            continue

        values = np.asarray(bundle[key])

        if (
            values.ndim == 1
            and len(values) == n_documents
        ):
            labels = np.asarray(
                [
                    normalize_label_value(v)
                    for v in values
                ],
                dtype=object,
            )

            valid = pd.Series(labels).astype(str).str.lower()
            n_valid = (~valid.isin(["", "nan", "none", "null"])).sum()

            if n_valid and pd.Series(labels).nunique() >= 2:
                return {
                    "full_labels": labels,
                    "matched": int(n_valid),
                    "source_rows": int(n_documents),
                    "unmatched_source": 0,
                    "conflicts": 0,
                    "n_classes": int(pd.Series(labels).nunique()),
                    "match_rate_source": 1.0,
                    "strategy": f"feature_bundle:{key}",
                    "source": f"bundle:{key}",
                }

    return None


def candidate_score(path):
    low = str(path).lower()

    positive_tokens = [
        "annot",
        "sentiment",
        "sentimen",
        "label",
        "mbg",
        "dataset",
        "manual",
        "tweet",
    ]

    negative_tokens = [
        "/results/",
        "/figures/",
        "/checkpoints/",
        "experiment_08",
        "experiment_07",
        "stage8v2",
    ]

    positive = sum(token in low for token in positive_tokens)
    negative = sum(token in low for token in negative_tokens)

    # More positive tokens = earlier; fewer penalties = earlier.
    return (
        -positive,
        negative,
        len(str(path)),
    )


def collect_candidate_dataset_paths():
    candidates = []

    # 1. Project root first.
    roots = [
        PROJECT_ROOT,
        DRIVE_ROOT,
    ]

    patterns = [
        "*.csv",
        "*.xlsx",
        "*.xls",
        "*.json",
    ]

    for root in roots:
        if not root.exists():
            continue

        for pattern in patterns:
            try:
                candidates.extend(
                    root.rglob(pattern)
                )
            except Exception:
                pass

    unique = []

    seen = set()

    for path in sorted(
        set(candidates),
        key=candidate_score,
    ):
        key = str(path)

        if key in seen:
            continue
        seen.add(key)

        if is_excluded_label_path(path):
            continue

        try:
            if (
                not path.is_file()
                or path.stat().st_size == 0
                or path.stat().st_size > 750 * 1024 * 1024
            ):
                continue
        except Exception:
            continue

        unique.append(path)

    return unique


def try_candidate_dataset(path):
    try:
        df = read_tabular(path)

        if df is None or df.empty:
            return None

        df = normalize_columns(df)

        text_col = find_column(
            df,
            TEXT_COLUMNS,
        )

        label_col = find_column(
            df,
            LABEL_COLUMNS,
        )

        if (
            text_col is None
            or label_col is None
        ):
            return None

        alignment = align_annotated_subset_to_hmrnet(
            df,
            text_col,
            label_col,
        )

        if alignment is None:
            return None

        alignment["source"] = str(path)
        alignment["text_column"] = str(text_col)
        alignment["label_column"] = str(label_col)

        return alignment

    except Exception:
        return None


# ------------------------------------------------------------------
# AUTO-RECOVERY
# ------------------------------------------------------------------

label_result = labels_from_bundle()

if label_result is None:
    candidate_paths = collect_candidate_dataset_paths()

    print(
        f"[AUTO LABEL] Candidate tabular files discovered: "
        f"{len(candidate_paths)}"
    )

    successful = []

    # Keep several matches and choose the strongest alignment.
    for file_no, path in enumerate(
        candidate_paths,
        start=1,
    ):
        result = try_candidate_dataset(
            path
        )

        if result is not None:
            successful.append(
                result
            )

            print(
                "[AUTO LABEL MATCH]",
                f"matched={result['matched']}",
                f"classes={result['n_classes']}",
                f"rate={result['match_rate_source']:.3f}",
                path,
            )

        # If a near-complete, clearly good annotation set is found,
        # we can stop early.
        if (
            successful
            and successful[-1]["matched"] >= 5000
            and successful[-1]["match_rate_source"] >= 0.95
        ):
            break

    if successful:
        # Prioritize:
        # 1. largest number of aligned labeled documents;
        # 2. highest source match rate;
        # 3. fewer conflicts.
        successful = sorted(
            successful,
            key=lambda r: (
                -r["matched"],
                -r["match_rate_source"],
                r["conflicts"],
            ),
        )

        label_result = successful[0]


if label_result is None:
    raise RuntimeError(
        "\n"
        "AUTO-LABEL RECOVERY FAILED AGAIN.\n\n"
        "The corrected code now accepts a labeled subset and searches the "
        "entire /content/drive/MyDrive, but no tabular dataset containing "
        "both a text column and a sentiment/label column could be aligned "
        "to the current HMRNet feature bundle.\n\n"
        "This means the original ANNOTATED_PATH file is probably no longer "
        "present in Google Drive, has been renamed, or uses text that cannot "
        "be aligned to the current 23,370-document feature bundle.\n\n"
        "Do NOT generate replacement labels. Locate the original annotated "
        "MBG file used by Experiment 5 and place it anywhere in MyDrive, "
        "then rerun this cell."
    )


# ------------------------------------------------------------------
# Convert the recovered subset into a full-length mask.
# ------------------------------------------------------------------

labels = np.asarray(
    label_result[
        "full_labels"
    ],
    dtype=object,
)

if len(labels) != n_documents:
    raise RuntimeError(
        "Internal label alignment error: "
        f"{len(labels)} labels vs {n_documents} HMRNet documents."
    )

label_series = pd.Series(
    labels,
    dtype="object",
)

valid_mask = (
    label_series.notna()
    & ~label_series.astype(str).str.lower().isin(
        [
            "",
            "nan",
            "none",
            "null",
        ]
    )
)

matched_labels = (
    label_series[
        valid_mask
    ]
    .astype(str)
    .map(normalize_label_value)
)

# Write normalized labels back.
label_series.loc[
    valid_mask
] = matched_labels.values

n_labeled = int(
    valid_mask.sum()
)

class_counts = (
    label_series[
        valid_mask
    ]
    .value_counts()
)

if class_counts.size < 2:
    raise RuntimeError(
        "Recovered annotation subset contains fewer than two classes."
    )

# For an 80:20 stratified split, each class must have enough examples.
if (class_counts < 5).any():
    raise RuntimeError(
        "At least one recovered sentiment class has fewer than 5 matched "
        "documents, so a reliable stratified 80:20 split cannot be created.\n"
        f"Class counts:\n{class_counts}"
    )


label_source = str(
    label_result.get(
        "source",
        label_result.get(
            "strategy",
            "unknown",
        ),
    )
)

label_recovery_method = str(
    label_result[
        "strategy"
    ]
)

AUTO_LABEL_CACHE = (
    EXP5_DIR
    / "experiment_05_stage8v2_labels_reused_subset.npy"
)

AUTO_LABEL_METADATA = (
    EXP5_DIR
    / "experiment_05_stage8v2_label_source_subset.json"
)

np.save(
    AUTO_LABEL_CACHE,
    labels,
    allow_pickle=True,
)

label_metadata = {
    "recovery_method": label_recovery_method,
    "label_source": label_source,
    "text_column": label_result.get(
        "text_column"
    ),
    "label_column": label_result.get(
        "label_column"
    ),
    "n_hmrnet_documents": int(
        n_documents
    ),
    "n_source_annotation_rows": int(
        label_result.get(
            "source_rows",
            n_labeled,
        )
    ),
    "n_matched_labeled_documents": int(
        n_labeled
    ),
    "source_match_rate": float(
        label_result.get(
            "match_rate_source",
            np.nan,
        )
    ),
    "alignment_conflicts": int(
        label_result.get(
            "conflicts",
            0,
        )
    ),
    "n_classes": int(
        class_counts.size
    ),
    "class_distribution": {
        str(k): int(v)
        for k, v in class_counts.items()
    },
    "cache_file": str(
        AUTO_LABEL_CACHE
    ),
}

with open(
    AUTO_LABEL_METADATA,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        label_metadata,
        f,
        indent=2,
        ensure_ascii=False,
    )


print("\n" + "=" * 92)
print("EXPERIMENT-5 ANNOTATION SUBSET RECOVERED")
print("=" * 92)
print("Source              :", label_source)
print("Alignment strategy  :", label_recovery_method)
print(
    "Matched labeled docs:",
    n_labeled,
    "/",
    n_documents,
)
print(
    "Source match rate   :",
    f"{label_result.get('match_rate_source', np.nan):.4f}",
)
print(
    "Alignment conflicts :",
    label_result.get(
        "conflicts",
        0,
    ),
)
print("\nClass distribution:")
display(
    class_counts
    .rename_axis("label")
    .to_frame("count")
)

print("\nSaved provenance:")
print(AUTO_LABEL_METADATA)

## 9. Prepare labeled features

In [ ]:
labeled_idx = np.where(
    valid_mask.to_numpy()
)[0]

if len(labeled_idx) == 0:
    raise RuntimeError(
        "No aligned labeled documents are available for Experiment 5."
    )

X_transformer = semantic_embeddings[
    labeled_idx
]

X_hmrnet = hmrnet_topic_matrix[
    labeled_idx
]

y_raw = (
    label_series.iloc[
        labeled_idx
    ]
    .astype(str)
    .map(normalize_label_value)
    .to_numpy()
)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(
    y_raw
)

print(
    "Experiment-5 labeled subset:",
    len(y),
    "documents",
)
print(
    "Classes:",
    list(
        label_encoder.classes_
    ),
)
print(
    "Transformer:",
    X_transformer.shape,
)
print(
    "HMRNet topics:",
    X_hmrnet.shape,
)

print("\nEncoded class distribution:")
display(
    pd.Series(
        y_raw
    )
    .value_counts()
    .rename_axis(
        "label"
    )
    .to_frame(
        "count"
    )
)

np.save(
    EXP5_DIR
    / "labeled_indices.npy",
    labeled_idx,
)

np.save(
    EXP5_DIR
    / "labels_encoded.npy",
    y,
)

## 10. Discover previous baseline topic representations

Notebook mencari matrix `(N, d)` yang namanya mengandung `lda`, `bertopic`, atau `top2vec` serta kata seperti `topic`, `theta`, `representation`, `embedding`, atau `document`.

Matrix hanya diterima jika jumlah baris sama dengan jumlah dokumen HMRNet.

In [ ]:
def try_load_numeric_matrix(path):
    try:
        suffix = path.suffix.lower()

        if suffix == ".npy":
            obj = np.load(path, allow_pickle=True)

        elif suffix == ".npz":
            npz = np.load(path, allow_pickle=True)
            arrays = [
                np.asarray(npz[k])
                for k in npz.files
                if np.asarray(npz[k]).ndim == 2
            ]
            if not arrays:
                return None
            obj = arrays[0]

        elif suffix in {".joblib", ".pkl", ".pickle"}:
            obj = joblib.load(path)

            if isinstance(obj, dict):
                preferred = [
                    "theta",
                    "document_topics",
                    "topic_distribution",
                    "topic_matrix",
                    "representations",
                    "embeddings",
                    "X",
                ]
                values = []
                for key in preferred:
                    if key in obj:
                        values.append(obj[key])
                values.extend(obj.values())

                obj = next(
                    (
                        v for v in values
                        if hasattr(v, "shape")
                        and len(getattr(v, "shape", ())) == 2
                        and v.shape[0] == n_documents
                    ),
                    None,
                )

        else:
            return None

        if obj is None:
            return None

        if sparse.issparse(obj):
            arr = obj.toarray()
        else:
            arr = np.asarray(obj)

        if (
            arr.ndim == 2
            and arr.shape[0] == n_documents
            and 2 <= arr.shape[1] <= 5000
            and np.issubdtype(arr.dtype, np.number)
        ):
            return arr.astype(np.float32)

    except Exception:
        return None

    return None


def discover_topic_vector(model_name):
    model_token = model_name.lower().replace("-", "").replace("_", "")
    hints = ["topic", "theta", "repr", "embedding", "document", "prob"]

    candidates = []

    for suffix in ["*.npy", "*.npz", "*.joblib", "*.pkl", "*.pickle"]:
        for path in RESULTS_DIR.rglob(suffix):
            normalized = path.name.lower().replace("-", "").replace("_", "")
            if model_token in normalized and any(h in normalized for h in hints):
                candidates.append(path)

    # Prefer checkpoint/results artifacts over generic files.
    candidates = sorted(set(candidates), key=lambda p: len(str(p)))

    for path in candidates:
        arr = try_load_numeric_matrix(path)
        if arr is not None:
            print(f"{model_name}: loaded precomputed vectors from {path} -> {arr.shape}")
            return arr, str(path)

    print(f"{model_name}: no compatible precomputed document-topic matrix found.")
    return None, None


baseline_topic_matrices = {}
baseline_sources = {}

if AUTO_DISCOVER_BASELINE_VECTORS:
    for name in ["LDA", "BERTopic", "Top2Vec"]:
        matrix, source = discover_topic_vector(name)
        if matrix is not None:
            baseline_topic_matrices[name] = matrix
            baseline_sources[name] = source

## 11. LDA fallback if no previous LDA topic matrix is available

In [ ]:
if "LDA" not in baseline_topic_matrices and FIT_LDA_IF_MISSING:
    print("Fitting LDA fallback with K=20 on the fixed BoW matrix...")

    lda = LatentDirichletAllocation(
        n_components=20,
        learning_method="batch",
        max_iter=30,
        random_state=42,
        n_jobs=-1,
    )

    lda_theta = lda.fit_transform(bow)

    baseline_topic_matrices["LDA"] = lda_theta.astype(np.float32)
    baseline_sources["LDA"] = "refit:sklearn_LDA_K20_seed42"

    np.save(
        EXP5_DIR / "lda_refit_document_topic.npy",
        lda_theta.astype(np.float32),
    )

    print("LDA fallback:", lda_theta.shape)

## 12. Optional BERTopic fallback

In [ ]:
if "BERTopic" not in baseline_topic_matrices and RETRAIN_BERTOPIC_IF_MISSING:
    print("Retraining BERTopic fallback...")

    %pip install -q "bertopic==0.17.4"

    from bertopic import BERTopic

    topic_model = BERTopic(
        embedding_model=None,
        calculate_probabilities=True,
        verbose=True,
    )

    bert_topics, bert_probs = topic_model.fit_transform(
        raw_texts,
        embeddings=semantic_embeddings,
    )

    if bert_probs is not None:
        bert_probs = np.asarray(bert_probs)

    if (
        bert_probs is not None
        and bert_probs.ndim == 2
        and bert_probs.shape[0] == n_documents
    ):
        bert_matrix = bert_probs.astype(np.float32)
    else:
        # Deterministic one-hot fallback from cluster assignments.
        unique_topics = sorted(set(bert_topics))
        topic_to_col = {topic: i for i, topic in enumerate(unique_topics)}
        bert_matrix = np.zeros(
            (n_documents, len(unique_topics)),
            dtype=np.float32,
        )
        for i, topic in enumerate(bert_topics):
            bert_matrix[i, topic_to_col[topic]] = 1.0

    baseline_topic_matrices["BERTopic"] = bert_matrix
    baseline_sources["BERTopic"] = "refit:BERTopic_native"

    np.save(
        EXP5_DIR / "bertopic_refit_document_topic.npy",
        bert_matrix,
    )

    print("BERTopic fallback:", bert_matrix.shape)

## 13. Optional Top2Vec note

In [ ]:
if "Top2Vec" not in baseline_topic_matrices:
    if RETRAIN_TOP2VEC_IF_MISSING:
        print(
            "Top2Vec automatic retraining is intentionally not implemented here "
            "because its embedding/training configuration must match the original "
            "Experiment 01 for a defensible comparison."
        )
    else:
        print(
            "Top2Vec topic vectors were not found. "
            "The notebook will use the previous Experiment-05 metric row if available."
        )

## 14. Experiment 5 metric helpers

In [ ]:
def multiclass_auc(y_true, probabilities, n_classes):
    try:
        if n_classes == 2:
            return float(
                roc_auc_score(
                    y_true,
                    probabilities[:, 1],
                )
            )

        return float(
            roc_auc_score(
                y_true,
                probabilities,
                multi_class="ovr",
                average="macro",
            )
        )
    except Exception:
        return np.nan


def evaluate_classifier_features(
    X,
    y,
    train_idx,
    test_idx,
    seed,
):
    clf = LogisticRegression(
        max_iter=LOGREG_MAX_ITER,
        class_weight=LOGREG_CLASS_WEIGHT,
        random_state=seed,
        solver="lbfgs",
    )

    clf.fit(
        X[train_idx],
        y[train_idx],
    )

    pred = clf.predict(
        X[test_idx]
    )

    prob = clf.predict_proba(
        X[test_idx]
    )

    return {
        "Accuracy": accuracy_score(
            y[test_idx],
            pred,
        ),
        "Macro-F1": f1_score(
            y[test_idx],
            pred,
            average="macro",
        ),
        "Weighted-F1": f1_score(
            y[test_idx],
            pred,
            average="weighted",
        ),
        "ROC-AUC": multiclass_auc(
            y[test_idx],
            prob,
            n_classes=len(np.unique(y)),
        ),
    }


def concat_features(base, topic):
    return np.hstack(
        [
            np.asarray(base, dtype=np.float32),
            np.asarray(topic, dtype=np.float32),
        ]
    )

## 15. Experiment 5 — Fixed 80:20 split with final HMRNet

In [ ]:
primary_train_idx, primary_test_idx = train_test_split(
    np.arange(len(y)),
    test_size=TEST_SIZE,
    random_state=PRIMARY_SPLIT_SEED,
    shuffle=True,
    stratify=y,
)

feature_sets = {
    "Transformer only": X_transformer,
    "Transformer + HMRNet-Final": concat_features(
        X_transformer,
        X_hmrnet,
    ),
}

for name, matrix in baseline_topic_matrices.items():
    topic_labeled = matrix[labeled_idx]
    feature_sets[f"Transformer + {name}"] = concat_features(
        X_transformer,
        topic_labeled,
    )

primary_rows = []

for representation, X in feature_sets.items():
    print("Evaluating:", representation, X.shape)

    metrics = evaluate_classifier_features(
        X=X,
        y=y,
        train_idx=primary_train_idx,
        test_idx=primary_test_idx,
        seed=PRIMARY_SPLIT_SEED,
    )

    primary_rows.append({
        "Representation": representation,
        "Source": (
            "Stage8v2 final checkpoint"
            if representation == "Transformer + HMRNet-Final"
            else "Current rerun"
        ),
        **metrics,
    })

primary_df = pd.DataFrame(primary_rows).sort_values(
    "Macro-F1",
    ascending=False,
).reset_index(drop=True)

display(primary_df.round(4))

primary_df.to_excel(
    TABLES_DIR / "experiment_05_stage8v2_primary.xlsx",
    index=False,
)
primary_df.to_csv(
    TABLES_DIR / "experiment_05_stage8v2_primary.csv",
    index=False,
)

## 16. Recover unchanged baseline rows from previous Experiment 05 when needed

In [ ]:
def find_previous_experiment05_table():
    candidates = []

    for pattern in [
        "*experiment*05*.xlsx",
        "*experiment*05*.csv",
        "*downstream*.xlsx",
        "*downstream*.csv",
    ]:
        candidates.extend(TABLES_DIR.glob(pattern))
        candidates.extend(RESULTS_DIR.glob(pattern))

    # Exclude new Stage8v2 outputs.
    candidates = [
        p for p in set(candidates)
        if "stage8v2" not in p.name.lower()
    ]

    for path in sorted(candidates, key=lambda p: len(str(p))):
        try:
            df = (
                pd.read_excel(path)
                if path.suffix.lower() == ".xlsx"
                else pd.read_csv(path)
            )

            lower = {str(c).lower(): c for c in df.columns}

            if (
                any("macro" in k and "f1" in k for k in lower)
                and any("representation" in k or "model" in k for k in lower)
            ):
                print("Previous Experiment-05 table:", path)
                return df, path
        except Exception:
            continue

    return None, None


previous_exp5_df, previous_exp5_path = find_previous_experiment05_table()

comparison_df = primary_df.copy()

if previous_exp5_df is not None:
    df = previous_exp5_df.copy()

    # Normalize likely column names.
    rename_map = {}
    for col in df.columns:
        lc = str(col).lower().replace("_", "-")
        if "representation" in lc or lc == "model":
            rename_map[col] = "Representation"
        elif "accuracy" in lc:
            rename_map[col] = "Accuracy"
        elif "macro" in lc and "f1" in lc:
            rename_map[col] = "Macro-F1"
        elif "weighted" in lc and "f1" in lc:
            rename_map[col] = "Weighted-F1"
        elif "roc" in lc and "auc" in lc:
            rename_map[col] = "ROC-AUC"

    df = df.rename(columns=rename_map)

    needed = [
        "Representation",
        "Accuracy",
        "Macro-F1",
        "Weighted-F1",
        "ROC-AUC",
    ]

    if all(c in df.columns for c in needed):
        old = df[needed].copy()
        old["Representation"] = old["Representation"].astype(str)

        # Keep only baseline topic rows not already rerun.
        current_names = set(comparison_df["Representation"].astype(str))

        baseline_old = old[
            ~old["Representation"].isin(current_names)
            & ~old["Representation"].str.contains(
                "HMRNet",
                case=False,
                na=False,
            )
        ].copy()

        if len(baseline_old):
            baseline_old["Source"] = (
                f"Previous Experiment 05 unchanged baseline: {previous_exp5_path.name}"
            )

            comparison_df = pd.concat(
                [
                    comparison_df,
                    baseline_old[
                        [
                            "Representation",
                            "Source",
                            "Accuracy",
                            "Macro-F1",
                            "Weighted-F1",
                            "ROC-AUC",
                        ]
                    ],
                ],
                ignore_index=True,
            )

comparison_df = comparison_df.sort_values(
    "Macro-F1",
    ascending=False,
).reset_index(drop=True)

print("\nFinal comparison table:")
display(comparison_df.round(4))

comparison_df.to_excel(
    TABLES_DIR / "experiment_05_stage8v2_comparison.xlsx",
    index=False,
)

## 17. Experiment 5 — 10 repeated stratified splits for Q1 uncertainty

In [ ]:
repeated_rows = []

if RUN_REPEATED_SPLITS:
    repeated_feature_sets = {
        "Transformer only": X_transformer,
        "Transformer + HMRNet-Final": concat_features(
            X_transformer,
            X_hmrnet,
        ),
    }

    # Include topic baselines only when actual document-topic vectors are available.
    for name, matrix in baseline_topic_matrices.items():
        repeated_feature_sets[
            f"Transformer + {name}"
        ] = concat_features(
            X_transformer,
            matrix[labeled_idx],
        )

    for seed in REPEATED_SPLIT_SEEDS:
        tr_idx, te_idx = train_test_split(
            np.arange(len(y)),
            test_size=TEST_SIZE,
            random_state=seed,
            shuffle=True,
            stratify=y,
        )

        for representation, X in repeated_feature_sets.items():
            metrics = evaluate_classifier_features(
                X=X,
                y=y,
                train_idx=tr_idx,
                test_idx=te_idx,
                seed=seed,
            )

            repeated_rows.append({
                "Seed": seed,
                "Representation": representation,
                **metrics,
            })

repeated_df = pd.DataFrame(repeated_rows)

if len(repeated_df):
    repeated_summary_df = (
        repeated_df
        .groupby("Representation")
        .agg(
            Accuracy_Mean=("Accuracy", "mean"),
            Accuracy_Std=("Accuracy", "std"),
            Macro_F1_Mean=("Macro-F1", "mean"),
            Macro_F1_Std=("Macro-F1", "std"),
            Weighted_F1_Mean=("Weighted-F1", "mean"),
            Weighted_F1_Std=("Weighted-F1", "std"),
            ROC_AUC_Mean=("ROC-AUC", "mean"),
            ROC_AUC_Std=("ROC-AUC", "std"),
        )
        .reset_index()
        .sort_values("Macro_F1_Mean", ascending=False)
    )

    display(repeated_summary_df.round(4))

    repeated_df.to_excel(
        TABLES_DIR / "experiment_05_stage8v2_repeated_runs.xlsx",
        index=False,
    )
    repeated_summary_df.to_excel(
        TABLES_DIR / "experiment_05_stage8v2_repeated_summary.xlsx",
        index=False,
    )
else:
    repeated_summary_df = pd.DataFrame()
    print("Repeated splits disabled.")

## 18. Paired statistical tests: HMRNet-Final vs Transformer-only

In [ ]:
stat_rows = []

if len(repeated_df):
    pivot = repeated_df.pivot(
        index="Seed",
        columns="Representation",
        values="Macro-F1",
    )

    baseline_name = "Transformer only"
    hmr_name = "Transformer + HMRNet-Final"

    if baseline_name in pivot.columns and hmr_name in pivot.columns:
        x = pivot[baseline_name].dropna()
        z = pivot[hmr_name].dropna()

        common = x.index.intersection(z.index)
        x = x.loc[common].to_numpy()
        z = z.loc[common].to_numpy()

        differences = z - x

        try:
            w_stat, w_p = wilcoxon(
                z,
                x,
                alternative="two-sided",
                zero_method="wilcox",
            )
        except Exception:
            w_stat, w_p = np.nan, np.nan

        t_stat, t_p = ttest_rel(
            z,
            x,
            nan_policy="omit",
        )

        stat_rows.append({
            "Comparison": f"{hmr_name} vs {baseline_name}",
            "N_Paired_Splits": len(common),
            "Transformer_MacroF1_Mean": float(np.mean(x)),
            "HMRNet_MacroF1_Mean": float(np.mean(z)),
            "Mean_Delta_MacroF1": float(np.mean(differences)),
            "Median_Delta_MacroF1": float(np.median(differences)),
            "Wilcoxon_Statistic": float(w_stat) if np.isfinite(w_stat) else np.nan,
            "Wilcoxon_p": float(w_p) if np.isfinite(w_p) else np.nan,
            "Paired_t_Statistic": float(t_stat) if np.isfinite(t_stat) else np.nan,
            "Paired_t_p": float(t_p) if np.isfinite(t_p) else np.nan,
        })

statistical_df = pd.DataFrame(stat_rows)

if len(statistical_df):
    display(statistical_df.round(6))
    statistical_df.to_excel(
        TABLES_DIR / "experiment_05_stage8v2_significance.xlsx",
        index=False,
    )

## 19. Experiment 5 visualization

In [ ]:
plot_df = comparison_df.dropna(
    subset=["Macro-F1"]
).sort_values(
    "Macro-F1",
    ascending=True,
)

plt.figure(figsize=(9, 5))
plt.barh(
    plot_df["Representation"],
    plot_df["Macro-F1"],
)
plt.xlabel("Macro-F1")
plt.title("Experiment 5 — Downstream Utility with HMRNet Stage 8 v2")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "experiment_05_stage8v2_macro_f1.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# Experiment 6 — Automatic Semantic Interpretability

## 20. Normalize final HMRNet topics

In [ ]:
raw_topics = hmrnet.get_topics(
    top_n=TOP_N_WORDS
)

def normalize_topic_words(topics, top_n):
    normalized = []

    if isinstance(topics, dict):
        iterable = [
            topics[k]
            for k in sorted(topics)
        ]
    else:
        iterable = list(topics)

    for topic in iterable:
        words = []

        if isinstance(topic, dict):
            topic = list(topic.items())

        for item in topic:
            if isinstance(item, str):
                word = item
            elif isinstance(item, (tuple, list)) and len(item) >= 1:
                word = item[0]
            else:
                word = str(item)

            word = str(word).strip()

            if word:
                words.append(word)

            if len(words) >= top_n:
                break

        normalized.append(words)

    return normalized

topic_words = normalize_topic_words(
    raw_topics,
    TOP_N_WORDS,
)

K = len(topic_words)

if K != hmrnet_topic_matrix.shape[1]:
    print(
        "WARNING: number of extracted topics differs from topic matrix:",
        K,
        "vs",
        hmrnet_topic_matrix.shape[1],
    )

print("Number of topics:", K)
for k, words in enumerate(topic_words):
    print(f"Topic {k:02d}:", ", ".join(words))

## 21. Load semantic model for automatic interpretability

In [ ]:
semantic_model = SentenceTransformer(
    SEMANTIC_MODEL_NAME,
    device=DEVICE,
)

print("Semantic interpretability model:", SEMANTIC_MODEL_NAME)

unique_words = sorted(
    {
        word
        for words in topic_words
        for word in words
    }
)

word_vectors = semantic_model.encode(
    unique_words,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

word_vector_map = {
    word: vec
    for word, vec in zip(
        unique_words,
        word_vectors,
    )
}

print("Unique topic words:", len(unique_words))
print("Embedding dim:", word_vectors.shape[1])

## 22. Topic centroids and Semantic Compactness

In [ ]:
topic_centroids = []
compactness_rows = []

for k, words in enumerate(topic_words):
    vectors = np.vstack(
        [word_vector_map[word] for word in words]
    )

    centroid = vectors.mean(axis=0)
    norm = np.linalg.norm(centroid)

    if norm > 0:
        centroid = centroid / norm

    topic_centroids.append(centroid)

    similarities = (
        vectors @ centroid
    )

    compactness_rows.append({
        "Topic": k,
        "N_Words": len(words),
        "Semantic_Compactness": float(
            np.mean(similarities)
        ),
        "Min_Word_to_Centroid": float(
            np.min(similarities)
        ),
        "Max_Word_to_Centroid": float(
            np.max(similarities)
        ),
        "Top_Words": ", ".join(words),
    })

topic_centroids = np.vstack(topic_centroids)
compactness_df = pd.DataFrame(compactness_rows)

MEAN_SC = float(
    compactness_df[
        "Semantic_Compactness"
    ].mean()
)

print("Mean Semantic Compactness:", MEAN_SC)
display(compactness_df.round(4))

compactness_df.to_excel(
    TABLES_DIR / "experiment_06_stage8v2_semantic_compactness.xlsx",
    index=False,
)

## 23. Inter-topic Separation

In [ ]:
centroid_similarity = cosine_similarity(
    topic_centroids
)

upper_idx = np.triu_indices(
    K,
    k=1,
)

mean_intertopic_cosine = float(
    centroid_similarity[
        upper_idx
    ].mean()
)

INTER_TOPIC_SEPARATION = float(
    1.0 - mean_intertopic_cosine
)

print("Mean inter-topic cosine:", mean_intertopic_cosine)
print("Inter-topic Separation:", INTER_TOPIC_SEPARATION)

np.save(
    EXP6_DIR / "topic_centroid_similarity.npy",
    centroid_similarity,
)

pd.DataFrame(
    centroid_similarity
).to_excel(
    TABLES_DIR / "experiment_06_stage8v2_topic_similarity_matrix.xlsx",
    index=False,
)

## 24. Word Intrusion Score (repeated deterministic test)

In [ ]:
rng_master = np.random.default_rng(
    WIS_RANDOM_SEED
)

wis_rows = []

for repeat in range(WIS_REPEATS):
    repeat_seed = int(
        rng_master.integers(
            0,
            2**31 - 1,
        )
    )
    rng = np.random.default_rng(
        repeat_seed
    )

    for k, words in enumerate(topic_words):
        other_topics = [
            j
            for j in range(K)
            if j != k
        ]

        rng.shuffle(other_topics)

        intruder_word = None
        intruder_topic = None

        for j in other_topics:
            candidates = [
                w
                for w in topic_words[j]
                if w not in words
            ]

            if candidates:
                intruder_word = str(
                    rng.choice(candidates)
                )
                intruder_topic = j
                break

        if intruder_word is None:
            continue

        candidate_words = list(words) + [
            intruder_word
        ]

        vectors = np.vstack(
            [
                word_vector_map[word]
                for word in candidate_words
            ]
        )

        target_vectors = np.vstack(
            [
                word_vector_map[word]
                for word in words
            ]
        )

        centroid = target_vectors.mean(
            axis=0
        )

        norm = np.linalg.norm(
            centroid
        )

        if norm > 0:
            centroid = centroid / norm

        similarities = (
            vectors @ centroid
        )

        predicted_idx = int(
            np.argmin(similarities)
        )

        predicted_word = (
            candidate_words[
                predicted_idx
            ]
        )

        correct = int(
            predicted_word
            == intruder_word
        )

        wis_rows.append({
            "Repeat": repeat + 1,
            "Repeat_Seed": repeat_seed,
            "Topic": k,
            "Intruder_Topic": intruder_topic,
            "Intruder_Word": intruder_word,
            "Predicted_Intruder": predicted_word,
            "Correct": correct,
        })

wis_detail_df = pd.DataFrame(
    wis_rows
)

wis_by_repeat_df = (
    wis_detail_df
    .groupby("Repeat")
    ["Correct"]
    .mean()
    .rename("WIS")
    .reset_index()
)

WIS_MEAN = float(
    wis_by_repeat_df[
        "WIS"
    ].mean()
)

WIS_STD = float(
    wis_by_repeat_df[
        "WIS"
    ].std(ddof=1)
)

print(
    f"WIS = {WIS_MEAN:.4f} ± {WIS_STD:.4f} "
    f"over {WIS_REPEATS} repeats"
)

display(
    wis_by_repeat_df.round(4)
)

wis_detail_df.to_excel(
    TABLES_DIR / "experiment_06_stage8v2_word_intrusion_detail.xlsx",
    index=False,
)

wis_by_repeat_df.to_excel(
    TABLES_DIR / "experiment_06_stage8v2_word_intrusion_repeats.xlsx",
    index=False,
)

## 25. Topic Intrusion Score

Reference topic = dominant HMRNet topic for each document.

For each document, the candidate set contains:
- its dominant HMRNet topic;
- `TIS_N_DISTRACTORS` randomly selected distractor topics.

The semantic topic is selected using cosine similarity between the document embedding and the topic centroid. TIS is the proportion for which the semantic choice recovers the HMRNet dominant topic.

In [ ]:
if INTERPRETABILITY_DOC_LIMIT is None:
    tis_indices = np.arange(n_documents)
else:
    rng = np.random.default_rng(TIS_RANDOM_SEED)
    limit = min(
        int(INTERPRETABILITY_DOC_LIMIT),
        n_documents,
    )
    tis_indices = np.sort(
        rng.choice(
            n_documents,
            size=limit,
            replace=False,
        )
    )

# Ensure document embeddings share the same semantic space.
if semantic_embeddings.shape[1] == topic_centroids.shape[1]:
    tis_doc_embeddings = semantic_embeddings[
        tis_indices
    ].astype(np.float32)

    norms = np.linalg.norm(
        tis_doc_embeddings,
        axis=1,
        keepdims=True,
    )
    tis_doc_embeddings = (
        tis_doc_embeddings
        / np.maximum(norms, 1e-12)
    )
else:
    print(
        "Semantic embedding dimension mismatch. "
        "Re-encoding raw texts with interpretability model..."
    )

    tis_doc_embeddings = semantic_model.encode(
        [
            raw_texts[i]
            for i in tis_indices
        ],
        batch_size=128,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

theta_tis = hmrnet_topic_matrix[
    tis_indices,
    :K,
]

dominant_topics = np.argmax(
    theta_tis,
    axis=1,
)

rng = np.random.default_rng(
    TIS_RANDOM_SEED
)

tis_correct = np.zeros(
    len(tis_indices),
    dtype=np.int8,
)

tis_predicted = np.zeros(
    len(tis_indices),
    dtype=np.int32,
)

for local_i, true_topic in enumerate(dominant_topics):
    possible = [
        k
        for k in range(K)
        if k != int(true_topic)
    ]

    n_distractors = min(
        TIS_N_DISTRACTORS,
        len(possible),
    )

    distractors = rng.choice(
        possible,
        size=n_distractors,
        replace=False,
    ).tolist()

    candidates = [
        int(true_topic)
    ] + [
        int(x)
        for x in distractors
    ]

    sims = (
        topic_centroids[candidates]
        @ tis_doc_embeddings[local_i]
    )

    predicted_topic = candidates[
        int(np.argmax(sims))
    ]

    tis_predicted[
        local_i
    ] = predicted_topic

    tis_correct[
        local_i
    ] = int(
        predicted_topic
        == int(true_topic)
    )

TIS = float(
    tis_correct.mean()
)

print("Documents evaluated:", len(tis_indices))
print("TIS:", TIS)

tis_detail_df = pd.DataFrame({
    "Document_Index": tis_indices,
    "HMRNet_Dominant_Topic": dominant_topics,
    "Semantic_Selected_Topic": tis_predicted,
    "Correct": tis_correct,
})

tis_detail_df.to_csv(
    TABLES_DIR / "experiment_06_stage8v2_topic_intrusion_detail.csv",
    index=False,
)

## 26. Experiment 6 final summary

In [ ]:
interpretability_summary_df = pd.DataFrame([
    {
        "Model": "HMRNet-Stage8-v2-Final",
        "Checkpoint": str(
            FINAL_HMRNET_CHECKPOINT
        ),
        "N_Topics": K,
        "Top_Words_Per_Topic": TOP_N_WORDS,
        "WIS_Mean": WIS_MEAN,
        "WIS_Std": WIS_STD,
        "WIS_Repeats": WIS_REPEATS,
        "TIS": TIS,
        "TIS_N_Documents": len(tis_indices),
        "TIS_Distractors": TIS_N_DISTRACTORS,
        "Semantic_Compactness": MEAN_SC,
        "Inter_Topic_Separation": INTER_TOPIC_SEPARATION,
        "Semantic_Model": SEMANTIC_MODEL_NAME,
    }
])

display(
    interpretability_summary_df.T.rename(
        columns={0: "Value"}
    )
)

interpretability_summary_df.to_excel(
    TABLES_DIR / "experiment_06_stage8v2_interpretability_summary.xlsx",
    index=False,
)
interpretability_summary_df.to_csv(
    TABLES_DIR / "experiment_06_stage8v2_interpretability_summary.csv",
    index=False,
)

## 27. Compare Experiment 6 with previous HMRNet-Full result

In [ ]:
previous_interpretability = pd.DataFrame([
    {
        "Model": "HMRNet-Full (previous)",
        "WIS": 0.1600,
        "TIS": 0.4330,
        "Semantic_Compactness": 0.7674,
        "Inter_Topic_Separation": 0.1333,
    },
    {
        "Model": "HMRNet-Stage8-v2-Final",
        "WIS": WIS_MEAN,
        "TIS": TIS,
        "Semantic_Compactness": MEAN_SC,
        "Inter_Topic_Separation": INTER_TOPIC_SEPARATION,
    },
])

previous_interpretability["Delta_WIS_vs_Previous"] = (
    previous_interpretability["WIS"] - 0.1600
)
previous_interpretability["Delta_TIS_vs_Previous"] = (
    previous_interpretability["TIS"] - 0.4330
)
previous_interpretability["Delta_SC_vs_Previous"] = (
    previous_interpretability["Semantic_Compactness"] - 0.7674
)
previous_interpretability["Delta_IS_vs_Previous"] = (
    previous_interpretability["Inter_Topic_Separation"] - 0.1333
)

display(
    previous_interpretability.round(4)
)

previous_interpretability.to_excel(
    TABLES_DIR / "experiment_06_stage8v2_comparison_previous.xlsx",
    index=False,
)

## 28. Interpretability visualization

In [ ]:
metric_names = [
    "WIS",
    "TIS",
    "Semantic Compactness",
    "Inter-topic Separation",
]

metric_values = [
    WIS_MEAN,
    TIS,
    MEAN_SC,
    INTER_TOPIC_SEPARATION,
]

plt.figure(figsize=(9, 5))
plt.bar(
    metric_names,
    metric_values,
)
plt.ylim(
    0,
    max(
        1.0,
        max(metric_values) * 1.1,
    ),
)
plt.ylabel("Score")
plt.title("Experiment 6 — HMRNet Stage 8 v2 Automatic Interpretability")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "experiment_06_stage8v2_interpretability.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 29. Save experiment metadata

In [ ]:
metadata = {
    "HMRNet_checkpoint": str(
        FINAL_HMRNET_CHECKPOINT
    ),
    "N_documents": int(
        n_documents
    ),
    "N_labeled_documents": int(
        len(y)
    ),
    "Label_source": str(
        label_source
    ),
    "Label_recovery_method": str(
        label_recovery_method
    ),
    "Label_policy": "Recovered automatically from the previous annotated subset; no new labels generated.",
    "Experiment_05": {
        "primary_split_seed": PRIMARY_SPLIT_SEED,
        "test_size": TEST_SIZE,
        "class_weight": LOGREG_CLASS_WEIGHT,
        "repeated_split_seeds": (
            REPEATED_SPLIT_SEEDS
            if RUN_REPEATED_SPLITS
            else []
        ),
        "baseline_topic_sources": baseline_sources,
    },
    "Experiment_06": {
        "top_n_words": TOP_N_WORDS,
        "semantic_model": SEMANTIC_MODEL_NAME,
        "wis_repeats": WIS_REPEATS,
        "tis_distractors": TIS_N_DISTRACTORS,
        "tis_n_documents": int(
            len(tis_indices)
        ),
    },
}

with open(
    TABLES_DIR / "experiment_05_06_stage8v2_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

print(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
        default=str,
    )
)

## 30. Verify expected outputs

In [ ]:
expected_outputs = [
    EXP5_DIR / "experiment_05_stage8v2_labels_reused_subset.npy",
    EXP5_DIR / "experiment_05_stage8v2_label_source_subset.json",
    TABLES_DIR / "experiment_05_stage8v2_primary.xlsx",
    TABLES_DIR / "experiment_05_stage8v2_comparison.xlsx",
    TABLES_DIR / "experiment_05_stage8v2_repeated_runs.xlsx",
    TABLES_DIR / "experiment_05_stage8v2_repeated_summary.xlsx",
    TABLES_DIR / "experiment_05_stage8v2_significance.xlsx",
    TABLES_DIR / "experiment_06_stage8v2_interpretability_summary.xlsx",
    TABLES_DIR / "experiment_06_stage8v2_semantic_compactness.xlsx",
    TABLES_DIR / "experiment_06_stage8v2_word_intrusion_repeats.xlsx",
    TABLES_DIR / "experiment_06_stage8v2_topic_intrusion_detail.csv",
    TABLES_DIR / "experiment_06_stage8v2_comparison_previous.xlsx",
    TABLES_DIR / "experiment_05_06_stage8v2_metadata.json",
    FIGURES_DIR / "experiment_05_stage8v2_macro_f1.png",
    FIGURES_DIR / "experiment_06_stage8v2_interpretability.png",
]

print("OUTPUT VERIFICATION")
print("=" * 90)

missing = []

for path in expected_outputs:
    exists = path.exists()
    print(("✓" if exists else "✗"), path)
    if not exists:
        missing.append(path)

if missing:
    print(
        "\nBeberapa output belum terbentuk. "
        "Jika RUN_REPEATED_SPLITS=False, output repeated/significance memang tidak dibuat."
    )
else:
    print("\n✓ Semua output utama Experiment 5 dan Experiment 6 tersedia.")